## Extraction of Data from MODIS Vegetation Indices Dataset (GEE) ##

In [2]:
# Import Google Earth Engine API and Initialize it. 
import ee
import pandas as pd

ee.Authenticate()
ee.Initialize(project="ey-data-and-ai-challenge")

In [3]:
# Read coordinates and date from water quality training dataset, drop given features.

wq_df = pd.read_csv('../data/water_quality_training_dataset.csv')
wq_df = wq_df.drop(columns=['Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus'])
wq_df['id'] = wq_df.index
wq_df['Sample Date'] = pd.to_datetime(wq_df['Sample Date'], format="%d-%m-%Y").dt.strftime("%Y-%m-%d")
wq_df.head()

,Latitude,Longitude,Sample Date,id
0,-28.760833,17.730278,2011-01-02,0
1,-26.861111,28.884722,2011-01-03,1
2,-26.450000,28.085833,2011-01-03,2
3,-27.671111,27.236944,2011-01-03,3
4,-27.356667,27.286389,2011-01-03,4


In [14]:
# Convert Coordinates and given date to ee.Features for use in batch export.

features = []

for index, row in wq_df.iterrows():
    feat = ee.Feature(
        ee.Geometry.Point([row['Longitude'], row['Latitude']]).buffer(250), #add a 100m buffer in case of inexact coordinates
        {'id': row['id'],
         'start_date': (pd.to_datetime(row['Sample Date']) - pd.Timedelta(weeks=4)).strftime('%Y-%m-%d'),
         'end_date': (pd.to_datetime(row['Sample Date']) + pd.Timedelta(weeks=4)).strftime('%Y-%m-%d')
        }
    )
    features.append(feat)

fc = ee.FeatureCollection(features)         # create feature collection with features

In [15]:
modisvi_collection = ee.ImageCollection("MODIS/061/MOD13Q1").select(['NDVI', 'EVI'])       # Selecting vegetation indices

In [16]:
def extract_median_values(feat):
    collection = modisvi_collection.filterDate(feat.get('start_date'), feat.get('end_date'))
    img = collection.reduce(ee.Reducer.median()) # reduce image collection into a single image

    modisvi_col = img.reduceRegions(collection=ee.FeatureCollection([feat]), reducer=ee.Reducer.first(), scale = 250)
    
    return modisvi_col.first()

In [17]:
fc_mapped = fc.map(extract_median_values)

In [18]:
# Process data and export to Google Drive

task = ee.batch.Export.table.toDrive(
    collection=fc_mapped,
    description="modisVI_csv_export",
    fileNamePrefix= "modisVI_features_training",
    fileFormat='CSV'
)
task.start()

In [19]:
modis_df = pd.read_csv("../data/modisVI_features_training.csv")

# Drop irrelevant columns
modis_df.drop(columns=[".geo", "system:index", "end_date", "start_date"], inplace=True)

modis_df = modis_df.merge(wq_df, on='id', how='left')
modis_df.drop(columns=['id'], inplace=True)
modis_df

,EVI_median,NDVI_median,Latitude,Longitude,Sample Date
0,241.0,636.0,-28.760833,17.730278,2011-01-02
1,4643.0,7656.0,-26.861111,28.884722,2011-01-03
2,3984.0,6276.0,-26.450000,28.085833,2011-01-03
3,2217.0,3957.0,-27.671111,27.236944,2011-01-03
4,4136.0,7396.0,-27.356667,27.286389,2011-01-03
...,...,...,...,...,...
9314,2783.0,5093.0,-27.527500,30.858056,2015-12-23
9315,3951.0,5845.5,-26.861111,28.884722,2015-12-23
9316,1744.0,2921.5,-26.984722,26.632278,2015-12-23
9317,2115.0,3917.5,-27.935000,26.126667,2015-12-23


In [20]:
modis_df.to_csv("../data/modisVI_features_training.csv")